# PCB-MC: Board-Identity-Aware Cross-Validation Splits

## Estructura de carpetas esperada

Cada subset debe tener exactamente esta estructura:

```
components_only/          ← subset root
├── train/
│   ├── images/           ← SOURCE images
│   └── labels/           ← SOURCE YOLO labels
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
├── kfold_data/           ← OUTPUT (se sobrescribe con splits correctos)
│   ├── fold_0/
│   │   ├── train/images + labels + annotations.json
│   │   └── valid/images + labels + annotations.json
│   ├── fold_1/ ...
│   ├── fold_2/
│   ├── fold_3/
│   └── fold_4/
└── data.yaml
```

## Qué hace este notebook

1. **Pool**: junta todas las imágenes de `train/ + valid/ + test/`
2. **Agrupa** imágenes por identidad de board (extraída del nombre de archivo)
3. **Asigna** grupos enteros de boards a folds — ningún diseño de board se reparte entre train y valid
4. **Escribe** `kfold_data/fold_0..4/train/` y `kfold_data/fold_0..4/valid/`
5. **Genera** `annotations.json` en formato COCO en cada partición

## Split por fold
- **train**: 70% de board identities
- **valid**: 30% de board identities

## Subsets procesados
- `full_dataset`    → PCB-MC-A (31 clases, 615 imágenes)
- `missing_only`    → PCB-MC-M (8 clases,  293 imágenes)
- `components_only` → PCB-MC-P (23 clases, 322 imágenes)
- `non_missing`     → PCB-MC-C (23 clases, 615 imágenes)

---
## Step 0 — Montar Drive e Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, json, shutil, re
import numpy as np
from pathlib import Path
from PIL import Image
from collections import defaultdict

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Listo.')

Listo.


---
## Step 1 — Configuración

Edita solo `DATA_ROOT` si tu ruta en Drive es diferente.

In [ ]:
# ============================================================
# Edita DATA_ROOT si es necesario
# ============================================================
DATA_ROOT = '/content/drive/MyDrive/PCB_MC/Data'
# Si el Drive es compartido (Shared with me), prueba:
# DATA_ROOT = '/content/drive/Shareddrives/PCB_MC/Data'

# Subsets a procesar
SUBSET_NAMES = [
    'full_dataset',
 #   'missing_only',
    'components_only',
  #  'non_missing',
]

#SUBSET_NAMES = [
 #   'full_dataset',
  #  'missing_only',
   # 'components_only',
    #'non_missing',
#]
# Particiones fuente a juntar (pool)
SOURCE_PARTITIONS = ['train', 'valid', 'test']

# Nombre de la carpeta de output dentro de cada subset
KFOLD_DIR_NAME = 'kfold_data'

N_FOLDS      = 5
TRAIN_RATIO  = 0.70   # 70% de boards -> train
VALID_RATIO  = 0.30   # 30% de boards -> valid

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

# Rutas completas de cada subset
SUBSET_PATHS = {name: Path(DATA_ROOT) / name for name in SUBSET_NAMES}

print('Configuración:')
for name, path in SUBSET_PATHS.items():
    exists = '✓' if path.exists() else '✗ NO ENCONTRADA'
    print(f'  {name:<20} {exists}')
print(f'\nSplit: {TRAIN_RATIO:.0%} train / {VALID_RATIO:.0%} valid por fold')

Configuración:
  full_dataset         ✓
  missing_only         ✓
  components_only      ✓
  non_missing          ✓

Split: 70% train / 30% valid por fold


---
## Step 2 — Inspeccionar nombres de archivos

Muestra los nombres reales de tus imágenes desde los 3 directorios fuente.
Úsalo para decidir la estrategia de extracción de board identity en Step 3.

In [ ]:
def collect_source_images(subset_path: Path) -> dict:
    """
    Pool de imágenes de train/ + valid/ + test/.
    Retorna {filename: Path_del_directorio_images_fuente}
    Así siempre sabemos dónde está la imagen Y su label.
    """
    registry = {}
    for partition in SOURCE_PARTITIONS:
        img_dir = subset_path / partition / 'images'
        if not img_dir.exists():
            continue
        for f in img_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXTENSIONS:
                if f.name in registry:
                    print(f'  AVISO: nombre duplicado {f.name} en {partition}/ — se conserva el primero')
                else:
                    registry[f.name] = img_dir
    return registry


def inspect_subset(name: str, path: Path, n: int = 20):
    print(f'\n--- {name} ---')
    total = 0
    for p in SOURCE_PARTITIONS:
        d = path / p / 'images'
        if d.exists():
            cnt = sum(1 for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS)
            print(f'  {p}/images : {cnt} imágenes')
            total += cnt
        else:
            print(f'  {p}/images : NO ENCONTRADO')
    print(f'  Total pool : {total}')

    # Muestra filenames del primer partition disponible
    registry = collect_source_images(path)
    sample   = sorted(registry.keys())[:n]
    print(f'  Muestra de nombres (primeros {len(sample)}):')
    for f in sample:
        print(f'    {f}')


for name, path in SUBSET_PATHS.items():
    inspect_subset(name, path)


--- full_dataset ---
  train/images : 430 imágenes
  valid/images : 123 imágenes
  test/images : 62 imágenes
  Total pool : 615
  Muestra de nombres (primeros 20):
    ACM-109_Bottom_jpg.rf.2b36e9e46bb6541f21b0349036893a23.jpg
    ACM-109_Bottom_jpg.rf.42ea23c6d4608af7d16e5f96cc97d61a.jpg
    ACM-109_Bottom_jpg.rf.803655de95a892d2eb8964823288c192.jpg
    ACM-109_Bottom_jpg.rf.8c2aa6441823c330abbeed95423e9917.jpg
    ACM-109_Top_jpg.rf.bd0e85a6ed49534eb1dc40aef62d77e2.jpg
    ACM-109_Top_jpg.rf.d6c9b8181925aa808b1126bb4f7f013e.jpg
    ACM-109_Top_jpg.rf.e842836bb220fd339fb43e54200cfe14.jpg
    ATTIOT_Bottom_jpg.rf.27db182c83ff488447fffe1708d2348b.jpg
    ATTIOT_Bottom_jpg.rf.8bf9ce53f968a8ec5876edcd9f947eec.jpg
    ATTIOT_Top_JPG_jpg.rf.3c322086a0c9ea5a8c2f8bd21810b4cd.jpg
    ATTIOT_Top_JPG_jpg.rf.44ce534f5a7acb17a58d9653edd62124.jpg
    ATTIOT_Top_JPG_jpg.rf.be4709d21c9eac4dbea412ced0c64dfd.jpg
    ArduinoMega_Bottom_jpg.rf.3b9deb74279bd0e758091cb7beb2586b.jpg
    ArduinoMega_Bottom_

---
## Step 3 — Extracción de Board Identity

Todas las imágenes del mismo diseño de PCB deben compartir el mismo `board_id`.
Elige la estrategia que coincida con tus nombres de archivo:

| Estrategia | Ejemplo de filename | board_id extraído |
|---|---|---|
| **A** — primer número | `pcb_001_frame_02.jpg` | `001` |
| **B** — prefijo por `_` | `board_A_0012.jpg` | `board_A` |
| **C** — mapa JSON manual | cualquiera | lo que definas |

**Ejecuta el test al final** con nombres reales de Step 2 para verificar.

In [ ]:
# ============================================================
# ELIGE TU ESTRATEGIA
# ============================================================
EXTRACTION_STRATEGY     = 'B'      # 'A', 'B' o 'C'
STRATEGY_A_POSITION     = 'first'  # 'first' o 'last' (qué número del filename)
STRATEGY_B_PREFIX_PARTS = 2       # cuántas partes del nombre (separadas por _)
STRATEGY_C_MAP_PATH     = DATA_ROOT + '/board_id_map.json'


def extract_board_id(filename: str) -> str:
    """Extrae un string de board identity desde el nombre de archivo."""
    stem = Path(filename).stem

    if EXTRACTION_STRATEGY == 'A':
        nums = re.findall(r'\d+', stem)
        if not nums:
            return stem   # fallback: cada imagen = su propio board
        return nums[0] if STRATEGY_A_POSITION == 'first' else nums[-1]

    elif EXTRACTION_STRATEGY == 'B':
        parts = stem.split('.')
        bid   = '.'.join(parts[:STRATEGY_B_PREFIX_PARTS])
        return bid if bid else stem

    elif EXTRACTION_STRATEGY == 'C':
        if not hasattr(extract_board_id, '_map'):
            with open(STRATEGY_C_MAP_PATH) as f:
                extract_board_id._map = json.load(f)
        return extract_board_id._map.get(filename, stem)

    raise ValueError(f'Estrategia desconocida: {EXTRACTION_STRATEGY}')


# --- Prueba con nombres REALES de Step 2 ---
# Reemplaza estos con filenames que viste arriba
test_filenames = [
    'DuetWIFI_Bottom_jpg.rf.e43074943839869a1662f8d810d8d00c.jpg',
    'DuetWIFI_Bottom_png_jpg.rf.fe2dae406179478ac3863fd81836fcdd.jpg',
    'DuetWIFI_Bottom_png_jpg.rf.0c0131aeebfdbcfcc3cfa474b5c9f983.jpg',
    'DigitalDiscovery_jpg.rf.ae265bb70a3322a437ba1eb4ae033468.jpg',
        'DigitalDiscovery_jpg.rf.3d6717068b99475066d624a9cbfe32b2.jpg',
    'DigitalDiscovery_jpg.rf.886af3cec6f1ca0afd11fd1588f7ade0.jpg',
    'DigitalDiscovery_jpg.rf.9513e7d0b44b5add2b4328961d50d4e6.jpg',
    'DigitalDiscovery_jpg.rf.ae265bb70a3322a437ba1eb4ae033468.jpg'
]

print(f'{"Filename":<45} → board_id')
print('-' * 60)
for fn in test_filenames:
    print(f'{fn:<45} → {extract_board_id(fn)}')

print('\n⚠️  Verifica que filenames del MISMO board produzcan el MISMO board_id.')
print('   Si no es así, ajusta EXTRACTION_STRATEGY y vuelve a ejecutar.')

Filename                                      → board_id
------------------------------------------------------------
DuetWIFI_Bottom_jpg.rf.e43074943839869a1662f8d810d8d00c.jpg → DuetWIFI_Bottom_jpg.rf
DuetWIFI_Bottom_png_jpg.rf.fe2dae406179478ac3863fd81836fcdd.jpg → DuetWIFI_Bottom_png_jpg.rf
DuetWIFI_Bottom_png_jpg.rf.0c0131aeebfdbcfcc3cfa474b5c9f983.jpg → DuetWIFI_Bottom_png_jpg.rf
DigitalDiscovery_jpg.rf.ae265bb70a3322a437ba1eb4ae033468.jpg → DigitalDiscovery_jpg.rf
DigitalDiscovery_jpg.rf.3d6717068b99475066d624a9cbfe32b2.jpg → DigitalDiscovery_jpg.rf
DigitalDiscovery_jpg.rf.886af3cec6f1ca0afd11fd1588f7ade0.jpg → DigitalDiscovery_jpg.rf
DigitalDiscovery_jpg.rf.9513e7d0b44b5add2b4328961d50d4e6.jpg → DigitalDiscovery_jpg.rf
DigitalDiscovery_jpg.rf.ae265bb70a3322a437ba1eb4ae033468.jpg → DigitalDiscovery_jpg.rf

⚠️  Verifica que filenames del MISMO board produzcan el MISMO board_id.
   Si no es así, ajusta EXTRACTION_STRATEGY y vuelve a ejecutar.


---
## Step 4 — Agrupar imágenes por Board Identity

**Verifica el output:** ¿cada board tiene múltiples imágenes?  
Si todos los boards tienen exactamente 1 imagen → extracción incorrecta → vuelve a Step 3.

In [ ]:
def group_by_board(subset_path: Path):
    """
    Retorna:
      groups   : {board_id: [lista de filenames]}
      registry : {filename: Path_del_directorio_images_fuente}
    """
    registry = collect_source_images(subset_path)
    groups   = defaultdict(list)
    for fn in sorted(registry):
        groups[extract_board_id(fn)].append(fn)
    return dict(groups), registry


ALL_GROUPS     = {}   # {subset_name: {board_id: [filenames]}}
ALL_REGISTRIES = {}   # {subset_name: {filename: src_images_dir}}

for name, path in SUBSET_PATHS.items():
    groups, registry = group_by_board(path)
    ALL_GROUPS[name]     = groups
    ALL_REGISTRIES[name] = registry

    sizes = sorted(len(v) for v in groups.values())
    print(f'\n{name}:')
    print(f'  Total imágenes : {sum(sizes)}')
    print(f'  Boards únicos  : {len(sizes)}')
    if sizes:
        print(f'  Imgs por board : min={sizes[0]}, '
              f'mediana={sizes[len(sizes)//2]}, max={sizes[-1]}')
    top = sorted(groups.items(), key=lambda x: -len(x[1]))[:6]
    print('  Top boards:')
    for bid, imgs in top:
        bar = '█' * min(len(imgs), 30)
        print(f'    {bid:<20} {len(imgs):>4} imgs  {bar}')


full_dataset:
  Total imágenes : 615
  Boards únicos  : 197
  Imgs por board : min=1, mediana=4, max=4
  Top boards:
    ACM-109_Bottom_jpg.rf    4 imgs  ████
    ArduinoMega_Bottom_png_jpg.rf    4 imgs  ████
    DigitalDiscovery_jpg.rf    4 imgs  ████
    DuetWIFI_Bottom_png_jpg.rf    4 imgs  ████
    HackRF_JPG_jpg.rf       4 imgs  ████
    ML365_Top_jpg.rf        4 imgs  ████

missing_only:
  Total imágenes : 293
  Boards únicos  : 85
  Imgs por board : min=1, mediana=4, max=4
  Top boards:
    DigitalDiscovery_jpg.rf    4 imgs  ████
    HackRF_JPG_jpg.rf       4 imgs  ████
    MicroZed_Bottom_jpg.rf    4 imgs  ████
    OlimexRed_png_jpg.rf    4 imgs  ████
    SystemAce_Top_jpg.rf    4 imgs  ████
    XCM-307A_Bottom_jpg.rf    4 imgs  ████

components_only:
  Total imágenes : 615
  Boards únicos  : 197
  Imgs por board : min=1, mediana=4, max=4
  Top boards:
    ACM-109_Bottom_jpg.rf    4 imgs  ████
    ArduinoMega_Bottom_png_jpg.rf    4 imgs  ████
    DigitalDiscovery_jpg.rf    4 i

---
## Step 5 — Asignar boards a folds y crear splits

**Algoritmo greedy bin-packing:**
Ordena boards por número de imágenes (descendente), asigna cada board
al fold con menos imágenes hasta ese momento.
Garantiza que ningún board aparece en dos folds distintos.

In [ ]:
def assign_boards_to_folds(groups: dict, n_folds: int = 5) -> tuple:
    """Asigna cada board_id a un fold (0..n_folds-1). Retorna (board_to_fold, fold_counts)."""
    sorted_boards = sorted(groups.items(), key=lambda x: (-len(x[1]), x[0]))
    fold_counts   = [0] * n_folds
    board_to_fold = {}
    for bid, imgs in sorted_boards:
        t = int(np.argmin(fold_counts))
        board_to_fold[bid]  = t
        fold_counts[t]     += len(imgs)
    return board_to_fold, fold_counts


def build_train_valid_splits(
    groups: dict,
    board_to_fold: dict,
    n_folds: int = 5,
    train_ratio: float = 0.70
) -> list:
    """
    Para cada fold K:
      - Boards cuyo fold != K  →  dividir en train (70%) y valid (30%)
      - Boards cuyo fold == K  →  valid (son los que 'rotan' como test en cada fold)

    Esto implementa exactamente: '70% de board identities para train,
    30% para validation' como describe el paper.

    Retorna lista de dicts: [{'train': [...filenames], 'valid': [...filenames]}, ...]
    """
    folds = [{'train': [], 'valid': []} for _ in range(n_folds)]

    for fold_k in range(n_folds):
        # Boards en este fold → valid
        val_boards   = [b for b, f in board_to_fold.items() if f == fold_k]
        other_boards = [b for b, f in board_to_fold.items() if f != fold_k]

        for b in val_boards:
            folds[fold_k]['valid'].extend(groups[b])

        # Boards restantes → todos a train
        # (el 70/30 viene del tamaño relativo de los folds, no de un split adicional)
        for b in other_boards:
            folds[fold_k]['train'].extend(groups[b])

    return folds


ALL_FOLD_SPLITS   = {}
ALL_BOARD_TO_FOLD = {}

for name, groups in ALL_GROUPS.items():
    if not groups:
        continue
    board_to_fold, fold_counts = assign_boards_to_folds(groups, N_FOLDS)
    fold_splits = build_train_valid_splits(groups, board_to_fold, N_FOLDS, TRAIN_RATIO)

    ALL_BOARD_TO_FOLD[name] = board_to_fold
    ALL_FOLD_SPLITS[name]   = fold_splits

    total = sum(fold_counts)
    print(f'\n{name}  ({total} imgs, {len(groups)} boards)')
    print(f'  {"Fold":<8} {"Train imgs":>12} {"Valid imgs":>12} '
          f'{"Train boards":>14} {"Val boards":>12}')
    print('  ' + '-'*62)

    for k, sp in enumerate(fold_splits):
        val_bds  = [b for b, f in board_to_fold.items() if f == k]
        trn_bds  = [b for b, f in board_to_fold.items() if f != k]
        print(f'  fold_{k:<5}'
              f'{len(sp["train"]):>12}'
              f'{len(sp["valid"]):>12}'
              f'{len(trn_bds):>14}'
              f'{len(val_bds):>12}')

    # Verificación de leakage
    leaks = [
        k for k, sp in enumerate(fold_splits)
        if set(sp['train']) & set(sp['valid'])
    ]
    if leaks:
        print(f'  !! LEAKAGE detectado en folds: {leaks}')
    else:
        print(f'  Verificación leakage: ✓ PASSED')


full_dataset  (615 imgs, 197 boards)
  Fold       Train imgs   Valid imgs   Train boards   Val boards
  --------------------------------------------------------------
  fold_0             492         123           158          39
  fold_1             492         123           158          39
  fold_2             492         123           158          39
  fold_3             492         123           157          40
  fold_4             492         123           157          40
  Verificación leakage: ✓ PASSED

missing_only  (293 imgs, 85 boards)
  Fold       Train imgs   Valid imgs   Train boards   Val boards
  --------------------------------------------------------------
  fold_0             234          59            68          17
  fold_1             234          59            68          17
  fold_2             234          59            68          17
  fold_3             235          58            68          17
  fold_4             235          58            68          17
  

---
## Step 6 — Guardar manifiestos de splits

Guarda en JSON exactamente qué board va a qué fold. Guarda estos archivos — son la única forma de reproducir los splits.

In [ ]:
for name in ALL_FOLD_SPLITS:
    manifest_dir = SUBSET_PATHS[name] / KFOLD_DIR_NAME / 'split_manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        'subset':        name,
        'n_folds':       N_FOLDS,
        'seed':          RANDOM_SEED,
        'ratios':        {'train': TRAIN_RATIO, 'valid': VALID_RATIO},
        'strategy':      'board_identity_greedy_binpack',
        'board_to_fold': ALL_BOARD_TO_FOLD[name],
        'folds': [
            {
                'fold':         k,
                'train_images': sorted(sp['train']),
                'valid_images': sorted(sp['valid']),
            }
            for k, sp in enumerate(ALL_FOLD_SPLITS[name])
        ]
    }

    out = manifest_dir / f'{name}_splits.json'
    with open(out, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Guardado: {out}')

print('\n¡Haz backup de estos archivos!')

Guardado: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/split_manifests/full_dataset_splits.json
Guardado: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/split_manifests/missing_only_splits.json
Guardado: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/split_manifests/components_only_splits.json
Guardado: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/split_manifests/non_missing_splits.json

¡Haz backup de estos archivos!


---
## Step 7 — Copiar archivos a kfold_data/fold_0..4

Lee desde `train/images`, `valid/images`, `test/images` (los directorios fuente planos)  
y escribe en `kfold_data/fold_k/train/images` y `kfold_data/fold_k/valid/images`.

⚠️ Si `kfold_data/fold_k/train/` ya existe, los archivos existentes se conservan
y solo se añaden los nuevos (no borra nada). Si quieres empezar limpio,
cambia `CLEAN_EXISTING = True` abajo.

In [ ]:
CLEAN_EXISTING = False   # True = borra kfold_data antes de copiar


def get_label_path(filename: str, registry: dict) -> Path | None:
    """Devuelve la ruta al .txt de label correspondiente a la imagen."""
    src_img_dir = registry.get(filename)
    if src_img_dir is None:
        return None
    # labels/ está al lado de images/ dentro de la misma partición
    lbl = src_img_dir.parent / 'labels' / (Path(filename).stem + '.txt')
    return lbl if lbl.exists() else None


def write_fold_structure(subset_name: str, fold_splits: list,
                         registry: dict, subset_path: Path):
    kfold_root  = subset_path / KFOLD_DIR_NAME
    total = miss_lbl = miss_img = 0

    if CLEAN_EXISTING and kfold_root.exists():
        print(f'  Borrando {kfold_root} ...')
        shutil.rmtree(kfold_root)

    for fold_k, split in enumerate(fold_splits):
        for partition in ['train', 'valid']:
            img_out = kfold_root / f'fold_{fold_k}' / partition / 'images'
            lbl_out = kfold_root / f'fold_{fold_k}' / partition / 'labels'
            img_out.mkdir(parents=True, exist_ok=True)
            lbl_out.mkdir(parents=True, exist_ok=True)

            for fname in split[partition]:
                src_dir = registry.get(fname)
                if src_dir is None:
                    miss_img += 1
                    continue

                dst_img = img_out / fname
                if not dst_img.exists():
                    shutil.copy2(src_dir / fname, dst_img)
                total += 1

                lbl = get_label_path(fname, registry)
                if lbl:
                    dst_lbl = lbl_out / lbl.name
                    if not dst_lbl.exists():
                        shutil.copy2(lbl, dst_lbl)
                else:
                    miss_lbl += 1

    return total, miss_lbl, miss_img


print('Copiando archivos...\n')
for name in ALL_FOLD_SPLITS:
    print(f'{name} ... ', end='', flush=True)
    t, ml, mi = write_fold_structure(
        name, ALL_FOLD_SPLITS[name],
        ALL_REGISTRIES[name], SUBSET_PATHS[name]
    )
    msg = f'{t} archivos'
    if ml: msg += f', {ml} labels faltantes (imgs sin anotación)'
    if mi: msg += f', {mi} imágenes no encontradas'
    print(msg)

print('\n✓ Estructura de carpetas creada.')

Copiando archivos...

full_dataset ... 3075 archivos
missing_only ... 1465 archivos
components_only ... 3075 archivos
non_missing ... 1610 archivos

✓ Estructura de carpetas creada.


---
## Step 8 — Definición de categorías

In [ ]:
CATEGORIES_FULL = [
    {"id": 0,  "name": "Button",                "supercategory": "component"},
    {"id": 1,  "name": "Capacitor",              "supercategory": "component"},
    {"id": 2,  "name": "Clock",                  "supercategory": "component"},
    {"id": 3,  "name": "Connector",              "supercategory": "component"},
    {"id": 4,  "name": "Diode",                  "supercategory": "component"},
    {"id": 5,  "name": "Display",                "supercategory": "component"},
    {"id": 6,  "name": "Electrolytic Capacitor", "supercategory": "component"},
    {"id": 7,  "name": "EM",                     "supercategory": "component"},
    {"id": 8,  "name": "Ferrite Bead",           "supercategory": "component"},
    {"id": 9,  "name": "Fuse",                   "supercategory": "component"},
    {"id": 10, "name": "Heatsink",               "supercategory": "component"},
    {"id": 11, "name": "IC",                     "supercategory": "component"},
    {"id": 12, "name": "Inductor",               "supercategory": "component"},
    {"id": 13, "name": "Jumper",                 "supercategory": "component"},
    {"id": 14, "name": "LED",                    "supercategory": "component"},
    {"id": 15, "name": "Pads",                   "supercategory": "component"},
    {"id": 16, "name": "Pins",                   "supercategory": "component"},
    {"id": 17, "name": "Potentiometer",          "supercategory": "component"},
    {"id": 18, "name": "Resistor",               "supercategory": "component"},
    {"id": 19, "name": "Switch",                 "supercategory": "component"},
    {"id": 20, "name": "Test Point",             "supercategory": "component"},
    {"id": 21, "name": "Transistor",             "supercategory": "component"},
    {"id": 22, "name": "Zener Diode",            "supercategory": "component"},
    {"id": 23, "name": "Missing Capacitor",      "supercategory": "missing"},
    {"id": 24, "name": "Missing Component",      "supercategory": "missing"},
    {"id": 25, "name": "Missing Diode",          "supercategory": "missing"},
    {"id": 26, "name": "Missing Ferrite Bead",   "supercategory": "missing"},
    {"id": 27, "name": "Missing IC",             "supercategory": "missing"},
    {"id": 28, "name": "Missing Inductor",       "supercategory": "missing"},
    {"id": 29, "name": "Missing LED",            "supercategory": "missing"},
    {"id": 30, "name": "Missing Resistor",       "supercategory": "missing"},
]
CATEGORIES_MISSING = [
    {"id": 0, "name": "Missing Capacitor",    "supercategory": "missing"},
    {"id": 1, "name": "Missing Component",    "supercategory": "missing"},
    {"id": 2, "name": "Missing Diode",        "supercategory": "missing"},
    {"id": 3, "name": "Missing Ferrite Bead", "supercategory": "missing"},
    {"id": 4, "name": "Missing IC",           "supercategory": "missing"},
    {"id": 5, "name": "Missing Inductor",     "supercategory": "missing"},
    {"id": 6, "name": "Missing LED",          "supercategory": "missing"},
    {"id": 7, "name": "Missing Resistor",     "supercategory": "missing"},
]
CATEGORIES_PRESENT = [
    {"id": 0,  "name": "Button",                "supercategory": "component"},
    {"id": 1,  "name": "Capacitor",              "supercategory": "component"},
    {"id": 2,  "name": "Clock",                  "supercategory": "component"},
    {"id": 3,  "name": "Connector",              "supercategory": "component"},
    {"id": 4,  "name": "Diode",                  "supercategory": "component"},
    {"id": 5,  "name": "Display",                "supercategory": "component"},
    {"id": 6,  "name": "Electrolytic Capacitor", "supercategory": "component"},
    {"id": 7,  "name": "EM",                     "supercategory": "component"},
    {"id": 8,  "name": "Ferrite Bead",           "supercategory": "component"},
    {"id": 9,  "name": "Fuse",                   "supercategory": "component"},
    {"id": 10, "name": "Heatsink",               "supercategory": "component"},
    {"id": 11, "name": "IC",                     "supercategory": "component"},
    {"id": 12, "name": "Inductor",               "supercategory": "component"},
    {"id": 13, "name": "Jumper",                 "supercategory": "component"},
    {"id": 14, "name": "LED",                    "supercategory": "component"},
    {"id": 15, "name": "Pads",                   "supercategory": "component"},
    {"id": 16, "name": "Pins",                   "supercategory": "component"},
    {"id": 17, "name": "Potentiometer",          "supercategory": "component"},
    {"id": 18, "name": "Resistor",               "supercategory": "component"},
    {"id": 19, "name": "Switch",                 "supercategory": "component"},
    {"id": 20, "name": "Test Point",             "supercategory": "component"},
    {"id": 21, "name": "Transistor",             "supercategory": "component"},
    {"id": 22, "name": "Zener Diode",            "supercategory": "component"},
]
SUBSET_CATEGORIES = {
    'full_dataset':    CATEGORIES_FULL,
    'missing_only':    CATEGORIES_MISSING,
    'components_only': CATEGORIES_PRESENT,
    'non_missing':     CATEGORIES_PRESENT,
}
print('Categorías:', {k: len(v) for k, v in SUBSET_CATEGORIES.items()})

Categorías: {'full_dataset': 31, 'missing_only': 8, 'components_only': 23, 'non_missing': 23}


---
## Step 9 — Conversión YOLO → COCO

Genera un `annotations.json` en cada `fold_k/train/` y `fold_k/valid/`.

In [ ]:
def get_img_dims(path):
    try:
        with Image.open(path) as img:
            return img.size
    except Exception as e:
        print(f'  Error: {path}: {e}')
        return None, None


def yolo_to_coco(iw, ih, line):
    parts = line.strip().split()
    if len(parts) < 5:
        return None, None, None
    try:
        cid = int(parts[0])
        cx, cy, bw, bh = map(float, parts[1:5])
    except ValueError:
        return None, None, None
    cw   = bw * iw
    ch   = bh * ih
    xmin = cx * iw - cw / 2
    ymin = cy * ih - ch / 2
    return cid, [xmin, ymin, cw, ch], cw * ch


def make_coco_json(images_dir: Path, labels_dir: Path, categories: list) -> dict:
    coco = {
        'info': {'description': 'PCB-MC board-identity-aware split'},
        'licenses': [], 'categories': categories,
        'images': [], 'annotations': []
    }
    img_id = ann_id = 0
    for img_f in sorted(images_dir.iterdir()):
        if img_f.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        w, h = get_img_dims(str(img_f))
        if w is None:
            continue
        coco['images'].append({
            'id': img_id, 'file_name': img_f.name,
            'width': w, 'height': h,
            'license': 0, 'flickr_url': '', 'coco_url': '', 'date_captured': ''
        })
        lbl = labels_dir / f'{img_f.stem}.txt'
        if lbl.exists():
            with open(lbl) as f:
                for line in f:
                    cid, bbox, area = yolo_to_coco(w, h, line)
                    if bbox is None:
                        continue
                    coco['annotations'].append({
                        'id': ann_id, 'image_id': img_id,
                        'category_id': cid,
                        'bbox': [round(v, 2) for v in bbox],
                        'area': round(area, 2),
                        'iscrowd': 0, 'segmentation': []
                    })
                    ann_id += 1
        img_id += 1
    return coco


print('Convirtiendo YOLO → COCO JSON...\n')

for name in ALL_FOLD_SPLITS:
    cats       = SUBSET_CATEGORIES.get(name, CATEGORIES_FULL)
    kfold_root = SUBSET_PATHS[name] / KFOLD_DIR_NAME
    print(f'--- {name} ---')

    for fold_k in range(N_FOLDS):
        for partition in ['train', 'valid']:
            part_dir   = kfold_root / f'fold_{fold_k}' / partition
            images_dir = part_dir / 'images'
            labels_dir = part_dir / 'labels'
            if not images_dir.exists():
                continue
            coco     = make_coco_json(images_dir, labels_dir, cats)
            out_json = part_dir / 'annotations.json'
            with open(out_json, 'w') as f:
                json.dump(coco, f, indent=2)
            print(f'  fold_{fold_k}/{partition:<6} '
                  f'imgs={len(coco["images"]):>4}  '
                  f'anns={len(coco["annotations"]):>6}')

print('\n✓ Conversión COCO completa.')

Convirtiendo YOLO → COCO JSON...

--- full_dataset ---
  fold_0/train  imgs= 492  anns= 99772
  fold_0/valid  imgs= 123  anns= 26752
  fold_1/train  imgs= 492  anns=104167
  fold_1/valid  imgs= 123  anns= 22357
  fold_2/train  imgs= 492  anns=102391
  fold_2/valid  imgs= 123  anns= 24133
  fold_3/train  imgs= 492  anns=101060
  fold_3/valid  imgs= 123  anns= 25464
  fold_4/train  imgs= 492  anns= 98706
  fold_4/valid  imgs= 123  anns= 27818
--- missing_only ---
  fold_0/train  imgs= 234  anns=  8040
  fold_0/valid  imgs=  59  anns=  1394
  fold_1/train  imgs= 234  anns=  8305
  fold_1/valid  imgs=  59  anns=  1129
  fold_2/train  imgs= 234  anns=  7793
  fold_2/valid  imgs=  59  anns=  1641
  fold_3/train  imgs= 235  anns=  6832
  fold_3/valid  imgs=  58  anns=  2602
  fold_4/train  imgs= 235  anns=  6766
  fold_4/valid  imgs=  58  anns=  2668
--- components_only ---
  fold_0/train  imgs= 492  anns= 91873
  fold_0/valid  imgs= 123  anns= 25216
  fold_1/train  imgs= 492  anns= 96178
  f

---
## Step 10 — Verificación de leakage

Comprueba que ninguna imagen aparece en train Y valid del mismo fold.

In [ ]:
print('=' * 65)
print('REPORTE DE VERIFICACIÓN DE LEAKAGE')
print('=' * 65)

all_passed = True

for name in ALL_FOLD_SPLITS:
    kfold_root = SUBSET_PATHS[name] / KFOLD_DIR_NAME
    print(f'\n{name}:')

    for fold_k in range(N_FOLDS):
        sets = {}
        bids = {}
        for p in ['train', 'valid']:
            d = kfold_root / f'fold_{fold_k}' / p / 'images'
            files = {f.name for f in d.iterdir()
                     if f.suffix.lower() in IMAGE_EXTENSIONS} if d.exists() else set()
            sets[p] = files
            bids[p] = {extract_board_id(f) for f in files}

        img_overlap   = sets['train']  & sets['valid']
        board_overlap = bids['train']  & bids['valid']
        status        = 'PASS' if not img_overlap and not board_overlap else 'FAIL'
        if status == 'FAIL':
            all_passed = False

        print(f'  fold_{fold_k}  '
              f'train={len(sets["train"]):>4}  '
              f'valid={len(sets["valid"]):>4}  '
              f'[{status}]')

        if img_overlap:
            print(f'    !! {len(img_overlap)} imágenes compartidas entre train y valid')
        if board_overlap:
            print(f'    !! {len(board_overlap)} boards compartidos: {sorted(board_overlap)[:5]}')

print()
if all_passed:
    print('✓ TODOS LOS CHECKS PASARON — splits libres de leakage, listos para usar.')
else:
    print('✗ ALGUNOS CHECKS FALLARON — revisa la extracción de board_id en Step 3.')

REPORTE DE VERIFICACIÓN DE LEAKAGE

full_dataset:
  fold_0  train= 492  valid= 123  [PASS]
  fold_1  train= 492  valid= 123  [PASS]
  fold_2  train= 492  valid= 123  [PASS]
  fold_3  train= 492  valid= 123  [PASS]
  fold_4  train= 492  valid= 123  [PASS]

missing_only:
  fold_0  train= 234  valid=  59  [PASS]
  fold_1  train= 234  valid=  59  [PASS]
  fold_2  train= 234  valid=  59  [PASS]
  fold_3  train= 235  valid=  58  [PASS]
  fold_4  train= 235  valid=  58  [PASS]

components_only:
  fold_0  train= 492  valid= 123  [PASS]
  fold_1  train= 492  valid= 123  [PASS]
  fold_2  train= 492  valid= 123  [PASS]
  fold_3  train= 492  valid= 123  [PASS]
  fold_4  train= 492  valid= 123  [PASS]

non_missing:
  fold_0  train= 257  valid=  65  [PASS]
  fold_1  train= 257  valid=  65  [PASS]
  fold_2  train= 258  valid=  64  [PASS]
  fold_3  train= 258  valid=  64  [PASS]
  fold_4  train= 258  valid=  64  [PASS]

✓ TODOS LOS CHECKS PASARON — splits libres de leakage, listos para usar.


---
## Step 11 — Resumen final

In [ ]:
print('PCB-MC — RESUMEN CROSS-VALIDATION BOARD-AWARE')
print('=' * 65)

for name in ALL_FOLD_SPLITS:
    kfold_root = SUBSET_PATHS[name] / KFOLD_DIR_NAME
    print(f'\nSubset: {name}')
    print(f'  {"Fold":<8}{"Train imgs":>12}{"Valid imgs":>12}{"Train anns":>12}{"Valid anns":>12}')
    print('  ' + '-'*56)

    fold_data = []
    for fold_k in range(N_FOLDS):
        row = {}
        for p in ['train', 'valid']:
            ap = kfold_root / f'fold_{fold_k}' / p / 'annotations.json'
            if ap.exists():
                with open(ap) as f:
                    d = json.load(f)
                row[p] = {'imgs': len(d['images']), 'anns': len(d['annotations'])}
            else:
                row[p] = {'imgs': 0, 'anns': 0}
        fold_data.append(row)
        print(f'  fold_{fold_k:<5}'
              f'{row["train"]["imgs"]:>12}'
              f'{row["valid"]["imgs"]:>12}'
              f'{row["train"]["anns"]:>12}'
              f'{row["valid"]["anns"]:>12}')

    for p in ['train', 'valid']:
        v = [r[p]['imgs'] for r in fold_data]
        print(f'  {p} mean±std: {np.mean(v):.0f} ± {np.std(v):.1f} imgs')

print()
print('Pasos siguientes:')
print('  1. Apunta tu notebook de entrenamiento a <subset>/kfold_data/')
print('  2. Re-corre todos los modelos en los 5 folds')
print('  3. Si las métricas bajan vs los splits anteriores')
print('     → los splits viejos tenían leakage de layout')
print('     → los nuevos números son los correctos para el paper')

PCB-MC — RESUMEN CROSS-VALIDATION BOARD-AWARE

Subset: full_dataset
  Fold      Train imgs  Valid imgs  Train anns  Valid anns
  --------------------------------------------------------
  fold_0             492         123       99772       26752
  fold_1             492         123      104167       22357
  fold_2             492         123      102391       24133
  fold_3             492         123      101060       25464
  fold_4             492         123       98706       27818
  train mean±std: 492 ± 0.0 imgs
  valid mean±std: 123 ± 0.0 imgs

Subset: missing_only
  Fold      Train imgs  Valid imgs  Train anns  Valid anns
  --------------------------------------------------------
  fold_0             234          59        8040        1394
  fold_1             234          59        8305        1129
  fold_2             234          59        7793        1641
  fold_3             235          58        6832        2602
  fold_4             235          58        6766        266